# Paper Reader on Google Colab

Listen to research papers while the paragraph, figure, table, or equation being read is highlighted on the PDF, running on a free Colab GPU instead of your laptop.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (or better).

Run the cells in order. The first setup takes about 15 minutes (it builds llama.cpp with CUDA and downloads about 6.3 GB of models).

Colab runtimes are temporary. Turn on `USE_DRIVE` below to keep processed papers in Google Drive between sessions.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/YOUR_USER/paper-reader"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Keep processed papers and exported bundles in Google Drive (survives runtime resets):
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/paper-reader"  #@param {type:"string"}
#@markdown Kokoro voice (e.g. af_heart, am_michael, bf_emma, bm_george):
VOICE = "af_heart"  #@param {type:"string"}

In [ ]:
#@title 2. Get the code (and mount Drive)
import os, subprocess
REPO = "/content/paper-reader"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
%cd {REPO}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/papers", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/exports", exist_ok=True)
    os.environ["PR_DATA_DIR"] = f"{DRIVE_DIR}/papers"
    os.environ["PR_EXPORT_DIR"] = f"{DRIVE_DIR}/exports"
os.environ["PR_VOICE"] = VOICE
print("Papers stored in:", os.environ.get("PR_DATA_DIR", f"{REPO}/data/papers"))

In [ ]:
#@title 3. Install (≈15 min the first time; skipped steps are fast on re-runs)
!bash scripts/colab/setup_colab.sh

In [ ]:
#@title 4. Start Paper Reader and open it
!bash scripts/colab/start_colab.sh
from google.colab import output
#@markdown Opens in a new browser tab. Uncheck to show it inside the notebook instead.
NEW_TAB = True  #@param {type:"boolean"}
if NEW_TAB:
    output.serve_kernel_port_as_window(8000, anchor_text="Open Paper Reader")
else:
    output.serve_kernel_port_as_iframe(8000, height=900)

### Batch processing (optional)

Process many papers without the UI. Each finished paper is exported as a bundle (`.tar.gz`, MP3 audio) to `exports/` (or your Drive folder). On your Mac, import bundles with:

```bash
cd backend && uv run python batch.py import /path/to/bundles/
```

In [ ]:
#@title 5. Batch-process papers
#@markdown Space-separated arXiv IDs/URLs, or paths to PDFs/folders (e.g. on Drive):
PAPERS = "1706.03762 2010.11929"  #@param {type:"string"}
PARALLEL = 2  #@param {type:"integer"}
import shlex
!KEEP_SERVERS=1 bash scripts/gpu/batch.sh {" ".join(shlex.quote(p) for p in PAPERS.split())} --parallel {PARALLEL}

In [ ]:
#@title 6. Download exported bundles to this computer
import glob, os
from google.colab import files
exp = os.environ.get("PR_EXPORT_DIR", "/content/paper-reader/exports")
for f in sorted(glob.glob(f"{exp}/*.tar.gz")):
    print("Downloading", os.path.basename(f))
    files.download(f)

In [ ]:
#@title Logs (if something goes wrong)
!tail -n 30 data/logs/backend.log data/logs/llama-docling.log data/logs/llama-qwen.log

In [ ]:
#@title Stop everything
!bash scripts/colab/start_colab.sh stop